# 04 - Analisis Missing Value

## Objective
- Kuantifikasi sebaran missing value di semua kelompok fitur
- Cari tahu apakah missing value berkorelasi dengan fraud
- Tentukan strategi imputasi yang tepat per kolom

In [ ]:
from pathlib import Path
import warnings
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

warnings.filterwarnings("ignore")
plt.style.use("seaborn-v0_8-darkgrid")

DATA_DIR = Path("../data/interim")
TRAIN_PATH = DATA_DIR / "train_merged.parquet"
METADATA_DIR = Path("../data/metadata")

In [ ]:
train = pd.read_parquet(TRAIN_PATH)
print(f'Data loaded: {train.shape}')

In [ ]:
missing = train.isnull().sum()
missing_pct = (missing / len(train) * 100).sort_values(ascending=False)
cols_missing = missing_pct[missing_pct > 0]

print('Missing Value:')
print(f'   Total columns        : {len(train.columns)}')
print(f'   With missing values  : {len(cols_missing)} ({len(cols_missing)/len(train.columns)*100:.1f}%)')
print(f'   0% missing         : {(missing_pct==0).sum()} columns')
print(f'   1-10% missing      : {((missing_pct>0)&(missing_pct<=10)).sum()} columns')
print(f'   10-50% missing     : {((missing_pct>10)&(missing_pct<=50)).sum()} columns')
print(f'   50-90% missing     : {((missing_pct>50)&(missing_pct<=90)).sum()} columns')
print(f'   >90% missing       : {(missing_pct>90).sum()} columns')

In [ ]:
col_group = {
    "Transaction": ("TransactionID", "TransactionDT", "TransactionAmt", "ProductCD"),
    "Card": tuple(f"card{i}" for i in range(1, 7)),
    "Address": ("addr1", "addr2", "dist1", "dist2"),
    "Email": ("P_emaildomain", "R_emaildomain"),
    "Count": tuple(f"C{i}" for i in range(1, 15)),
    "TimeDelta": tuple(f"D{i}" for i in range(1, 16)),
    "Match": tuple(f"M{i}" for i in range(1, 10)),
    "Vesta": tuple(f"V{i}" for i in range(1, 340)),
    "Identity": tuple(f"id_{str(i).zfill(2)}" for i in range(1, 39)),
    "Device": ("DeviceType", "DeviceInfo"),
}

print('Average Missing Rate by Feature Group:')
for name, cols in col_group.items():
    valid_cols = [c for c in cols if c in train.columns]
    if valid_cols:
        total_missing = train[valid_cols].isna().sum().sum()
        total_cells = train[valid_cols].size
        avg_missing = (total_missing / total_cells) * 100
        print(f'   {name}: {avg_missing:.1f}%')

In [ ]:
top_missing = missing_pct.head(30)

fig, ax = plt.subplots(figsize=(12, 8))
top_missing.plot(kind='barh', ax=ax, color='steelblue')
ax.set_title('Top 30 Columns with Most Missing Values', fontsize=14)
ax.set_xlabel('Missing (%)')
ax.axvline(x=95, color='red', linestyle='--', label='95% — threshold to drop columns')
ax.axvline(x=75, color='orange', linestyle='--', label='75% — use indicators only')
ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
results = []
for col in cols_missing.index[:30]:
    mask_missing = train[col].isnull()
    fraud_missing = train.loc[mask_missing, 'isFraud'].mean() * 100
    fraud_present = train.loc[~mask_missing, 'isFraud'].mean() * 100

    results.append({
        'column': col,
        'missing_pct': missing_pct[col],
        'fraud_missing': fraud_missing,
        'fraud_present': fraud_present,
        'fraud_diff': fraud_missing - fraud_present
    })

missing_fraud_df = pd.DataFrame(results)
print('Fraud Rate: missing vs present values')
print(missing_fraud_df.sort_values('fraud_diff', ascending=False).head(15).to_string(index=False))

In [ ]:
top_diff = missing_fraud_df.nlargest(10, 'fraud_diff')

fig, ax = plt.subplots(figsize=(12, 6))
x = range(len(top_diff))
width = 0.35

ax.bar([i - width/2 for i in x], top_diff['fraud_present'], width, label='Present Value', color='#2ecc71')
ax.bar([i + width/2 for i in x], top_diff['fraud_missing'], width, label='Missing', color='#e74c3c')

ax.set_xticks(list(x))
ax.set_xticklabels(top_diff['column'], rotation=45, ha='right')
ax.set_ylabel('Fraud Rate (%)')
ax.set_title('Fraud Rate: Missing vs Present Values (Top 10 Differences)', fontsize=14)
ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
def get_imputation_strategy(missing_pct, dtype):
    """Determine imputation strategy based on missing percentage and data type."""
    if missing_pct == 0:
        return "No imputation needed"
    if missing_pct > 95:
        return "Drop column"
    if missing_pct > 75:
        return "Indicator only"
    is_numeric = pd.api.types.is_numeric_dtype(dtype)
    if missing_pct > 10:
        return "Median + Indicator" if is_numeric else "Mode + Indicator"
    return "Median" if is_numeric else "Mode"

recommendations = []
for col in train.columns:
    pct = missing_pct.get(col, 0)
    strategy = get_imputation_strategy(pct, train[col].dtype)
    recommendations.append({
        "column": col,
        "missing_pct": pct,
        "dtype": str(train[col].dtype),
        "strategy": strategy
    })

rec_df = pd.DataFrame(recommendations)
print("Imputation Strategy Summary:")
print(rec_df["strategy"].value_counts().to_string())

In [ ]:
METADATA_DIR.mkdir(parents=True, exist_ok=True)
rec_df.to_csv(METADATA_DIR / "missing_value_report.csv", index=False)

print(f"Saved to {METADATA_DIR / 'missing_value_report.csv'}")

print("Sample output:")
print(rec_df.head(10).to_string(index=False))